# Seven Millennium Problems — Native Lean No-Sorry Verification

This notebook verifies the external review package:

`7MP_ClayStrict_NativeLeanNoSorry_PSBigBig.zip`

**Seven Millennium Problems — Clay-Strict Native Lean No-Sorry Verification by PSBigBig**

Author: **PSBigBig**  
Repository: https://github.com/onestardao/WFGY

## Verification steps

1. Upload the package ZIP.
2. Install Lean through `elan`.
3. Extract the package and locate `lakefile.lean`.
4. Run `lake build`.
5. Run the included machine-verification script.
6. Package the verification logs and reports for external review.

Expected successful result:

```text
lake build: PASS
run_machine_verification.py: PASS
```

## Step 1 — Upload the package ZIP

Upload `7MP_ClayStrict_NativeLeanNoSorry_PSBigBig.zip` when prompted.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
zips = [name for name in uploaded if name.lower().endswith('.zip')]
if not zips:
    raise SystemExit('No ZIP uploaded.')
PACKAGE_ZIP = zips[0]
print('Uploaded:', PACKAGE_ZIP)
print('Size bytes:', Path(PACKAGE_ZIP).stat().st_size)

## Step 2 — Install Lean / elan

In [ ]:
from pathlib import Path
import subprocess, os

def run(cmd, cwd=None, check=False):
    print('\n$ ' + cmd)
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)
    return p

if not (Path.home() / '.elan/bin/elan').exists():
    run('curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y', check=True)

os.environ['PATH'] = str(Path.home() / '.elan/bin') + ':' + os.environ.get('PATH','')
run('elan --version', check=True)
run('lean --version')
run('lake --version')

## Step 3 — Extract package and locate Lean project

In [ ]:
from pathlib import Path
import zipfile, shutil
WORK = Path('/content/7mp_native_lean_verify')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(PACKAGE_ZIP, 'r') as z:
    z.extractall(WORK)
lakefiles = list(WORK.rglob('lakefile.lean'))
if not lakefiles:
    raise SystemExit('No lakefile.lean found.')
PROJECT_DIR = lakefiles[0].parent
print('Project directory:', PROJECT_DIR)
if (PROJECT_DIR/'lean-toolchain').exists():
    print('lean-toolchain:', (PROJECT_DIR/'lean-toolchain').read_text().strip())

## Step 4 — Run Lean build

In [ ]:
import subprocess, os, json, time
from pathlib import Path
REPORT_DIR = PROJECT_DIR / 'verification_results'
REPORT_DIR.mkdir(exist_ok=True)

def run_capture(name, cmd):
    print('\n' + '='*80)
    print(name)
    print('$', cmd)
    print('='*80)
    start=time.time()
    p=subprocess.run(cmd, shell=True, cwd=PROJECT_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=os.environ.copy())
    print(p.stdout)
    (REPORT_DIR/f'{name}.log').write_text(p.stdout, encoding='utf-8')
    return {'name':name,'cmd':cmd,'returncode':p.returncode,'seconds':round(time.time()-start,3),'log':str((REPORT_DIR/f'{name}.log').relative_to(PROJECT_DIR))}

results=[]
results.append(run_capture('lake_build','lake build'))
(REPORT_DIR/'lean_build_summary.json').write_text(json.dumps({'lake_build_pass':results[-1]['returncode']==0,'results':results},indent=2),encoding='utf-8')

## Step 5 — Run package machine verification gates

In [ ]:
results.append(run_capture('machine_verification','python3 scripts/run_machine_verification.py'))
summary={'overall_pass': all(r['returncode']==0 for r in results), 'results': results}
(REPORT_DIR/'verification_overall_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))

## Step 6 — Package and download verification results

In [ ]:
from google.colab import files
import shutil
RESULT_ZIP=Path('/content/7MP_NativeLeanNoSorry_Verification_Results.zip')
if RESULT_ZIP.exists(): RESULT_ZIP.unlink()
shutil.make_archive(str(RESULT_ZIP.with_suffix('')), 'zip', REPORT_DIR)
print('Created:', RESULT_ZIP)
files.download(str(RESULT_ZIP))